# 4. Interactive launch-site map with Folium
This notebook uses the geo dataset if available. The resulting map is interactive when opened in Jupyter.

In [ ]:
from pathlib import Path
import pandas as pd
import folium
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists(): ROOT = ROOT.parent
DATA = ROOT / "data"
geo_path = DATA / "spacex_launch_geo.csv"
if not geo_path.exists():
    raise FileNotFoundError("spacex_launch_geo.csv not found. Run src/download_data.py or place the course CSV in data/.")
geo = pd.read_csv(geo_path)
display(geo.head())
print("Columns:", geo.columns.tolist())

In [ ]:
def find_col(candidates):
    return next((c for c in candidates if c in geo.columns), None)
lat_col = find_col(["Lat", "Latitude", "latitude", "lat"])
lon_col = find_col(["Long", "Longitude", "longitude", "lon", "long"])
name_col = find_col(["Launch Site", "LaunchSite", "launch_site", "Name"])
if not lat_col or not lon_col:
    raise ValueError("Could not identify latitude/longitude columns. Inspect geo.columns and adjust the column names.")
points = geo.dropna(subset=[lat_col, lon_col]).copy()
center = [points[lat_col].mean(), points[lon_col].mean()]
launch_map = folium.Map(location=center, zoom_start=4, tiles="OpenStreetMap")
for _, row in points.iterrows():
    label = str(row[name_col]) if name_col else "Launch site"
    folium.Marker([row[lat_col], row[lon_col]], tooltip=label, popup=label).add_to(launch_map)
launch_map

In [ ]:
# Save an HTML copy that can be opened in a browser.
out = ROOT / "reports" / "launch_sites_map.html"
out.parent.mkdir(exist_ok=True)
launch_map.save(out)
print("Saved:", out)